# 予想外のインフレは政府債務をどれだけ軽くしたか――再現ノートブック

秋森弘「予想外のインフレは政府債務をどれだけ軽くしたか――債務比率の分解、部門別の損得、統合政府――」（『北星論集』、2027年3月刊行）の表・図・本文の数値を、公開統計から再現する。

- 実行方法：このファイルがあるフォルダ（`data/raw/` が見えるフォルダ）で、上から順にすべて実行する。外部ネットワークは使わない（データはすべて `data/raw/` に保存済み）。
- 最後の節「本文の数値との突合せ」で、論文本文に書かれた数値と、このノートブックの出力が一致するかを自動で判定する。
- 年度は会計年度（4月〜翌年3月）。「2025年度」は2025年4月〜2026年3月。債務比率は年度末の普通国債残高（額面）÷その年度の名目GDP。

## データ出所（`data/raw/`）

| ファイル | 内容 | 出所 |
|---|---|---|
| `gaku-mfy2622.csv`, `gaku-jfy2622.csv` | 名目・実質GDP（年度、連鎖） | 内閣府「国民経済計算」2026年4-6月期2次速報 |
| `vintage/gaku-*_FY20xx初出.csv` | 公表当時の名目・実質GDP（翌年6月の1-3月期2次速報） | 同、各年の1-3月期2次速報 |
| `jgb_general_bonds_balance_2002_2026.csv` | 普通国債残高（年度末、額面、兆円） | 財務省「最近20カ年間の年度末の国債残高の推移」（FY2002〜2006は2012年3月版） |
| `04_jgb_avg_coupon.csv` | 普通国債の利率加重平均（年度末） | 財務省「普通国債の利率加重平均の各年ごとの推移」 |
| `mitoshi_gdp_forecast.csv`, `01_mitoshi_cpi_forecast.csv` | 実質・名目GDP成長率、消費者物価の見通し | 内閣府「政府経済見通し」各年度版 |
| `esp_jan_forecast.csv` | 民間予想（年度の実質・名目成長率） | 日本経済研究センター「ESPフォーキャスト」1月調査 |
| `02_cpi_monthly.csv` | 消費者物価指数（総合、月次） | 総務省 |
| `06_tax_revenue_fy.csv` | 一般会計税収 | 財務省「税収の推移」 |
| `ff_dl_fof_fiscal-year_jp.csv` | 資金循環統計（年度末ストック） | 日本銀行「資金循環統計」 |

In [1]:
# 0. 準備
import io, csv, re, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
FIG = ROOT / "output" / "figures" if (ROOT / "output").exists() else ROOT / "図"; FIG.mkdir(parents=True, exist_ok=True)
TAB = ROOT / "output" / "tables" if (ROOT / "output").exists() else ROOT / "analysis"; TAB.mkdir(parents=True, exist_ok=True)
CHECKS = []                      # (番号, 内容, 計算値, 本文の値, 許容差)
def chk(label, got, exp, tol): CHECKS.append((label, float(got), float(exp), float(tol)))
print("データフォルダ:", RAW.relative_to(ROOT))

データフォルダ: data\raw


In [2]:
# 1. データの読み込み
def fy_series(path):                     # 内閣府の年度CSV（10億円）→ 兆円
    out = {}
    for r in csv.reader(open(path, encoding="cp932")):
        m = re.match(r"^(\d{4})/4-3", (r[0] or "").strip())
        if m and r[1].strip(): out[int(m.group(1))] = float(r[1].replace(",", "")) / 1e3
    return pd.Series(out).sort_index()
Y = fy_series(RAW / "gaku-mfy2622.csv"); R = fy_series(RAW / "gaku-jfy2622.csv")          # 名目・実質GDP（兆円）
B = pd.read_csv(RAW / "jgb_general_bonds_balance_2002_2026.csv", index_col=0).iloc[:, 0]   # 普通国債残高（兆円）
C = pd.read_csv(RAW / "04_jgb_avg_coupon.csv", index_col=0).iloc[:, 0] / 100               # 平均表面利率
F = pd.read_csv(RAW / "mitoshi_gdp_forecast.csv", index_col=0)                              # 政府経済見通し
T = pd.read_csv(RAW / "06_tax_revenue_fy.csv", index_col=0).iloc[:, 0]                      # 一般会計税収（兆円）
E = pd.read_csv(RAW / "esp_jan_forecast.csv", index_col=0)                                   # ESP（1月調査）
print(Y.loc[2021:2025].round(1).to_dict()); print(B.loc[2021:2025].round(1).to_dict())

{2021: 576.2, 2022: 591.6, 2023: 620.6, 2024: 645.1, 2025: 672.7}
{2021: 991.4, 2022: 1027.1, 2023: 1053.7, 2024: 1079.7, 2025: 1104.3}


## 2. 分析1：債務比率の分解（4節、表2、図2）

In [3]:
# 2. 債務比率 b = B / Y の年ごとの変化を、5つの要因に恒等的に分解する
#    Δb = w [ i − g^e − ε^π − ε^n ] + x ,  w = b_{t-1}/(1+g_t)
d = pd.DataFrame({"B": B, "Y": Y, "R": R, "C": C}).dropna().loc[2002:2025]
d["b"] = d["B"] / d["Y"]
d["g"] = d["Y"].pct_change(); d["n"] = d["R"].pct_change()
d["pi"] = (1 + d["g"]) / (1 + d["n"]) - 1                           # GDPデフレーター変化率（実績）
d["ge"] = F["nominal_gdp_growth_forecast_pct"] / 100; d["ne"] = F["real_gdp_growth_forecast_pct"] / 100
d["pie"] = (1 + d["ge"]) / (1 + d["ne"]) - 1                        # 見通しのデフレーター変化率
d["i"] = (d["C"] + d["C"].shift(1)) / 2                             # 年度中の平均利率（近似）
d["bprev"] = d["b"].shift(1)
d = d.dropna().loc[2003:]
d["db"] = d["b"] - d["bprev"]
d["w"] = d["bprev"] / (1 + d["g"])
d["e_pi"] = (d["pi"] - d["pie"]) * (1 + d["ne"])                    # 予想外の物価
d["e_n"] = (d["n"] - d["ne"]) * (1 + d["pi"])                       # 予想外の実質成長
assert ((d["e_pi"] + d["e_n"]) - (d["g"] - d["ge"])).abs().max() < 1e-12      # 分解の恒等性
comp = pd.DataFrame({"利払い": d["w"] * d["i"], "事前の名目成長": -d["w"] * d["ge"],
                     "予想外の物価": -d["w"] * d["e_pi"], "予想外の実質成長": -d["w"] * d["e_n"]}) * 100
comp["赤字等（残差）"] = d["db"] * 100 - comp.sum(axis=1)
comp["債務比率の変化"] = d["db"] * 100
def period(a, z): return comp.loc[a:z].sum()
t2 = pd.DataFrame({f"{a}〜{z}年度": period(a, z) for a, z in [(2003, 2021), (2022, 2025), (2003, 2025)]}).T
print("表2 債務比率の変化の要因分解（ポイント）"); print(t2.round(1).to_string())
print("\n債務比率(%)", (d["b"] * 100).loc[[2021, 2022, 2025]].round(1).to_dict(), " 2002年度末:", round(float(d.loc[2003, "bprev"] * 100), 1))
noshock = comp.loc[2022:2025, ["利払い", "事前の名目成長", "赤字等（残差）"]].sum().sum()
c_rev = -d["w"] * (d["pi"] - d["pie"]) * (1 + d["n"]) * 100                  # 交差項(π−π^e)(n−n^e)を物価側に配賦した場合
print("交差項を物価側に配賦した場合の2022〜25の物価寄与:", round(float(c_rev.loc[2022:2025].sum()), 4), " 現行:", round(float(comp.loc[2022:2025, '予想外の物価'].sum()), 4))
print("予想外の物価・成長を除いた2022〜25年度の変化:", round(noshock, 2))

表2 債務比率の変化の要因分解（ポイント）
              利払い  事前の名目成長  予想外の物価  予想外の実質成長  赤字等（残差）  債務比率の変化
2003〜2021年度  27.8    -50.4    11.5      25.0     78.8     92.7
2022〜2025年度   5.3    -18.7   -13.9       6.8     12.7     -7.9
2003〜2025年度  33.1    -69.2    -2.4      31.8     91.5     84.8

債務比率(%) {2021: 172.1, 2022: 173.6, 2025: 164.2}  2002年度末: 79.4
交差項を物価側に配賦した場合の2022〜25の物価寄与: -13.7713  現行: -13.911
予想外の物価・成長を除いた2022〜25年度の変化: -0.74


In [4]:
# 3. 反実仮想（表3）：2021年度末を起点に、2025年度末の債務比率
def counterfactual(elast, price_only=False, start=2021, end=2025):
    Yc, Bc = {start: Y[start]}, {start: B[start]}
    for t in range(start + 1, end + 1):
        gcf = (1 + d.loc[t, "n"]) * (1 + d.loc[t, "pie"]) - 1 if price_only else d.loc[t, "ge"]
        Yc[t] = Yc[t - 1] * (1 + gcf)
        x = B[t] - B[t - 1] * (1 + d.loc[t, "i"])                 # 観測された赤字等（兆円）
        extra = elast * (T[t] / Y[t]) * (Y[t] - Yc[t])            # 名目GDPの上振れ分の税収増（仮定）
        Bc[t] = Bc[t - 1] * (1 + d.loc[t, "i"]) + x + extra
    return Bc[end] / Yc[end] * 100
t3 = pd.DataFrame({"税収弾性値 0": [counterfactual(0), counterfactual(0, True)], "1": [counterfactual(1), counterfactual(1, True)],
                   "2": [counterfactual(2), counterfactual(2, True)]}, index=["物価と実質成長がともに見通しどおり", "物価のみ見通しどおり"])
print("表3 2025年度末の債務比率の反実仮想（%）。実績:", round(float(d.loc[2025, "b"] * 100), 1)); print(t3.round(1).to_string())
acc = comp.loc[2022:2025, ["予想外の物価", "予想外の実質成長"]].sum().sum()
print("弾性値0の差(実績−反実仮想):", round(float(d.loc[2025, "b"] * 100 - t3.iloc[0, 0]), 4), " 会計寄与の和:", round(float(acc), 4),
      " 物価のみの差:", round(float(d.loc[2025, "b"] * 100 - t3.iloc[1, 0]), 4), " 物価寄与:", round(float(comp.loc[2022:2025, '予想外の物価'].sum()), 4))

表3 2025年度末の債務比率の反実仮想（%）。実績: 164.2
                   税収弾性値 0      1      2
物価と実質成長がともに見通しどおり    171.3  172.2  173.2
物価のみ見通しどおり           178.1  180.6  183.0
弾性値0の差(実績−反実仮想): -7.1195  会計寄与の和: -7.1581  物価のみの差: -13.938  物価寄与: -13.911


## 3. 頑健性（5節、表4・表5）

In [5]:
# 4. 政府見通しの偏り（表4）：誤差の平均とNewey–West t値（ラグ2）
err = (d["pi"] - d["pie"]) * 100
def nw_t(x, L=2):
    m = x.mean(); u = x - m; n = len(x); s = (u**2).sum() / n
    for l in range(1, L + 1): s += 2 * (1 - l / (L + 1)) * (u[l:].values * u[:-l].values).sum() / n
    return m, m / np.sqrt(s / n)
bias = {}
for a, z in [(2003, 2021), (2022, 2025), (2003, 2025)]:
    bias[(a, z)] = nw_t(err.loc[a:z]) + (z - a + 1,)
t4 = pd.DataFrame({f"{a}〜{z}年度": {"年数": v[2], "平均(pt)": v[0], "t値": v[1]} for (a, z), v in bias.items()}).T
print("表4 政府見通しの誤差"); print(t4.round(2).to_string())

表4 政府見通しの誤差
               年数  平均(pt)    t値
2003〜2021年度  19.0   -0.52 -3.38
2022〜2025年度   4.0    2.09  6.64
2003〜2025年度  23.0   -0.07 -0.18


In [6]:
# 5. 予想の置き方（表5）
wt = d["w"]
def contrib(pi_exp, ne=None):
    ne = d["ne"] if ne is None else ne
    return -wt * (d["pi"] - pi_exp) * (1 + ne) * 100
pi_all = (1 + Y.pct_change()) / (1 + R.pct_change()) - 1                  # 1995年度〜のデフレーター変化率（最新版）
naive1 = pi_all.shift(1).reindex(d.index)                                 # 前年度の最終実績（最新版）を予想とする機械的ベンチマーク
naive3 = pi_all.rolling(3).mean().shift(1).reindex(d.index)               # 過去3年度の平均（2004年度から有効）
alts = {"政府経済見通し": contrib(d["pie"]), "過去3年度の平均の実績": contrib(naive3), "前年度の実績": contrib(naive1)}
rows = {k: {"2004〜2021": v.loc[2004:2021].sum(), "2022〜2025": v.loc[2022:2025].sum(), "2004〜2025": v.loc[2004:2025].sum()} for k, v in alts.items()}
# ESP（2019年度以降）
e_ = d.loc[2019:2025].copy()
ne_esp = E["esp_real_growth_pct"] / 100; ge_esp = E["esp_nominal_growth_pct"] / 100
pie_esp = (1 + ge_esp) / (1 + ne_esp) - 1
c_esp = (-d["w"] * (d["pi"] - pie_esp) * (1 + ne_esp) * 100).loc[2019:2025]
c_gov = alts["政府経済見通し"].loc[2019:2025]
rows["民間予想（ESP）"] = {"2004〜2021": c_esp.loc[2019:2021].sum(), "2022〜2025": c_esp.loc[2022:2025].sum(), "2004〜2025": np.nan}
# 公表当時の実績（2015〜2025年度）
def vint(t):
    yy = f"{(t + 1) % 100:02d}"
    Yv = fy_series(RAW / "vintage" / f"gaku-mfy{yy}12_FY{t}初出.csv"); Rv = fy_series(RAW / "vintage" / f"gaku-jfy{yy}12_FY{t}初出.csv")
    g_, n_ = Yv[t] / Yv[t - 1] - 1, Rv[t] / Rv[t - 1] - 1
    return (1 + g_) / (1 + n_) - 1, Yv[t]
vv = {t: vint(t) for t in range(2015, 2026)}
pi_v = pd.Series({t: v[0] for t, v in vv.items()}); Yv_t = pd.Series({t: v[1] for t, v in vv.items()})
c_v2 = -(B.shift(1).reindex(Yv_t.index) / Yv_t) * (pi_v - d["pie"]) * (1 + d["ne"]) * 100        # 重み(B_{t-1}/Y_t)も初出のY_tにした場合
c_v = (-d["w"] * (pi_v - d["pie"]) * (1 + d["ne"]) * 100).loc[2015:2025]
c_l = alts["政府経済見通し"].loc[2015:2025]
rows["政府見通し・公表当時の実績"] = {"2004〜2021": c_v.loc[2015:2021].sum(), "2022〜2025": c_v.loc[2022:2025].sum(), "2004〜2025": np.nan}
t5 = pd.DataFrame(rows).T
print("表5 予想外の物価の寄与（債務比率のポイント）。ESP・公表当時の列の2004〜2021は、それぞれ2019〜2021年度、2015〜2021年度のみ")
print(t5.round(2).to_string())
print("\n重みも初出のY_tにした場合: 2022〜2025", round(float(c_v2.loc[2022:2025].sum()), 2), " 2015〜2021", round(float(c_v2.loc[2015:2021].sum()), 2))
print("\n参考：最新の実績で2015〜2021年度:", round(float(c_l.loc[2015:2021].sum()), 2), " 公表当時:", round(float(c_v.loc[2015:2021].sum()), 2))
rmse = {k: float(((d["pi"] - e).loc[2004:2021] * 100).pow(2).mean() ** 0.5) for k, e in {"政府": d["pie"], "前年実績": naive1, "3年平均": naive3}.items()}
print("RMSE(2004〜2021、同じ18年度):", {k: round(v, 2) for k, v in rmse.items()})
print("有効な年度数(2004〜2021):", {k: int(e.loc[2004:2021].notna().sum()) for k, e in {"政府": d["pie"], "前年実績": naive1, "3年平均": naive3}.items()})
print("税収の増加率 FY2020→25:", round(float(T[2025] / T[2020] - 1) * 100), "%  名目GDP:", round(float(Y[2025] / Y[2020] - 1) * 100), "%  対数比:", round(float(np.log(T[2025] / T[2020]) / np.log(Y[2025] / Y[2020])), 2))

表5 予想外の物価の寄与（債務比率のポイント）。ESP・公表当時の列の2004〜2021は、それぞれ2019〜2021年度、2015〜2021年度のみ
               2004〜2021  2022〜2025  2004〜2025
政府経済見通し            10.95     -13.91      -2.96
過去3年度の平均の実績        -5.24     -10.09     -15.33
前年度の実績             -1.80      -5.58      -7.38
民間予想（ESP）          -1.09     -11.25        NaN
政府見通し・公表当時の実績       7.55     -11.30        NaN

重みも初出のY_tにした場合: 2022〜2025 -11.67  2015〜2021 7.86

参考：最新の実績で2015〜2021年度: 3.77  公表当時: 7.55
RMSE(2004〜2021、同じ18年度): {'政府': 0.77, '前年実績': 0.87, '3年平均': 1.05}
有効な年度数(2004〜2021): {'政府': 18, '前年実績': 18, '3年平均': 18}
税収の増加率 FY2020→25: 38 %  名目GDP: 21 %  対数比: 1.69


## 4. 誰が負担したか（6節、表6・表7、図3・図4）

In [7]:
# 6. 資金循環統計（年度末ストック）の読み込み
fof = pd.read_csv(RAW / "ff_dl_fof_fiscal-year_jp.csv", encoding="cp932", header=None, low_memory=False)
yrs_f = [int(y) for y in fof.iloc[0, 3:]]
vals = fof.iloc[:, 3:].apply(pd.to_numeric, errors="coerce"); vals.columns = yrs_f; vals.index = fof[0].astype(str)
def G_(sec, side, inst):                                       # 億円
    k = f"FOF_FFYS{sec}{side}{inst}"
    return vals.loc[k].fillna(0) if k in vals.index else pd.Series(0.0, index=yrs_f)
def S(sec, side, inst): return G_(sec, side, inst) / 1e4        # 兆円
# 国債・財投債の保有シェア（資産側）
hold = {"日本銀行": S("110", "A", "311"), "預金取扱機関": S("120", "A", "311"), "保険・年金基金": S("130", "A", "311"),
        "家計等": S("430", "A", "311") + S("440", "A", "311"), "海外": S("500", "A", "311"),
        "政府・公的金融": S("420", "A", "311") + S("180", "A", "311")}
H = pd.DataFrame(hold); tot = S("700", "A", "311"); H["その他"] = tot - H.sum(axis=1)
sh = H.div(tot, axis=0)
print("保有シェア(%)"); print((sh.loc[[2002, 2012, 2020, 2025]] * 100).round(1).to_string())
# 消費者物価（総合）：年度平均の前年度比と政府見通し
m = pd.read_csv(RAW / "02_cpi_monthly.csv"); m["fy"] = m["ym"] // 100 - ((m["ym"] % 100) <= 3)
g_ = m.groupby("fy")["cpi_all_items_2025base"].agg(["mean", "count"]); g_ = g_[g_["count"] == 12]["mean"]
pi_c = g_.pct_change(); pie_c = pd.read_csv(RAW / "01_mitoshi_cpi_forecast.csv", index_col=0).iloc[:, 0] / 100
eps = ((pi_c - pie_c) / (1 + pi_c)).loc[2003:2025]              # 予想外のインフレ率 ε
prev = sh.shift(1).mul(B.reindex(sh.index).shift(1), axis=0).loc[2003:2025]      # 前年度末の保有額（額面換算、兆円）
Tr = prev.mul(eps, axis=0)                                        # 購買力の損失（正＝保有者の損）
t6 = pd.DataFrame({"2022〜2025年度": Tr.loc[2022:2025].sum(), "2003〜2021年度": Tr.loc[2003:2021].sum()})
t6.loc["合計"] = t6.sum()
print("\n表6 国債の直接保有者別の損失（兆円）"); print(t6.round(1).to_string())
cpi_bias = nw_t(((pi_c - pie_c) * 100).loc[2003:2021])
print("CPI見通しの誤差 2003〜2021: 平均", round(float(cpi_bias[0]), 3), " t値", round(float(cpi_bias[1]), 2))
priv = t6.loc[["預金取扱機関", "保険・年金基金", "家計等", "海外", "その他"], "2022〜2025年度"].sum()
print("民間と海外の直接保有者の損失:", round(float(priv), 1))
defl_gain = (d["bprev"] * Y.shift(1).reindex(d.index) * (d["pi"] - d["pie"]) / (1 + d["pi"])).loc[2022:2025].sum()
print("デフレーター基準の国の利得（国債のみ）:", round(float(defl_gain), 1), "  CPI基準の保有者の損失:", round(float(t6.loc['合計', '2022〜2025年度']), 1),
      "  比:", round(float(defl_gain / t6.loc['合計', '2022〜2025年度']), 2))

保有シェア(%)
      日本銀行  預金取扱機関  保険・年金基金  家計等   海外  政府・公的金融  その他
2002  15.1    32.4     21.2  2.7  2.9     19.7  6.1
2012  11.5    40.2     27.6  3.2  4.4      9.7  3.4
2020  48.3    13.6     23.7  1.5  6.5      4.0  2.4
2025  47.9    13.3     18.5  2.3  7.9      7.6  2.5



表6 国債の直接保有者別の損失（兆円）
         2022〜2025年度  2003〜2021年度
日本銀行            23.7        -17.0
預金取扱機関           5.6        -11.3
保険・年金基金         10.3        -12.1
家計等              0.7         -1.0
海外               3.1         -2.7
政府・公的金融          2.3         -3.0
その他              1.0         -1.2
合計              46.8        -48.2
CPI見通しの誤差 2003〜2021: 平均 -0.316  t値 -2.53
民間と海外の直接保有者の損失: 20.8
デフレーター基準の国の利得（国債のみ）: 83.6   CPI基準の保有者の損失: 46.8   比: 1.79


In [8]:
# 7. 名目ネット・ポジション（NNP）と部門別の損得（表7）
NOM = ["100", "200", "300", "500", "510", "520"]         # 現金・預金、貸出、債務証券、預け金、企業間・貿易信用、未収・未払金
def nnp(secs, insts): return sum(G_(s, "A", i) - G_(s, "L", i) for s in secs for i in insts) / 1e4
SEC = {"日本銀行": ["110"], "預金取扱機関": ["120"], "保険・年金基金": ["130"], "家計等": ["430", "440"],
       "民間非金融法人": ["411"], "一般政府": ["420"], "海外": ["500"]}
def build(insts):
    X = pd.DataFrame({k: nnp(v, insts) for k, v in SEC.items()})
    X["その他"] = nnp(["100", "400", "500"], insts) - X.sum(axis=1)       # 残差（その他金融、公的非金融法人など）
    return X
narrow, broad = build(NOM), build(NOM + ["400"])                           # 400＝保険・年金・定型保証
# 検算：各名目項目について、全部門の資産合計と負債合計が一致する（部門間の移転の総和がゼロ）
for i in NOM + ["400"]:
    assert abs(((G_("700", "A", i) - G_("700", "L", i)) / 1e4).loc[2002:2025]).max() < 1e-6, i
def gains(X): return -X.shift(1).loc[2003:2025].mul(eps, axis=0)
G1, G2 = gains(narrow), gains(broad)
t7 = pd.DataFrame({"NNP(2021年度末,狭義)": narrow.loc[2021], "損得(狭義)": G1.loc[2022:2025].sum(), "損得(広義)": G2.loc[2022:2025].sum(),
                   "2003〜21(狭義)": G1.loc[2003:2021].sum(), "2003〜21(広義)": G2.loc[2003:2021].sum()})
t7.loc["合計"] = t7.sum()
print("表7 予想外の消費者物価による部門別の損得（兆円、＋＝得）"); print(t7.round(1).to_string())
hh_parts = {"現金・預金": ["100"], "貸出": ["200"], "債務証券": ["300"], "その他(預け金・企業間信用・未収未払)": ["500", "510", "520"]}
hh_gain = {k: float((-nnp(["430", "440"], v).shift(1).loc[2003:2025] * eps).loc[2022:2025].sum()) for k, v in hh_parts.items()}
print("家計等の2022〜25の損得を項目別に(兆円):", {k: round(v, 2) for k, v in hh_gain.items()}, " 合計:", round(sum(hh_gain.values()), 2))
hh = {i: float((G_("430", "A", i) + G_("440", "A", i) - G_("430", "L", i) - G_("440", "L", i))[2025] / 1e4) for i in ["100", "200"]}
print("\n家計の2025年度末：現金・預金(資産−負債)", round(hh["100"]), " 貸出(借入)", round(hh["200"]))
boj = {"債務証券": float(S("110", "A", "300")[2025]), "貸出": float(S("110", "A", "200")[2025]), "日銀券等": float(S("110", "L", "110")[2025]),
       "日銀預け金": float(S("110", "L", "120")[2025])}
print("日銀の2025年度末:", {k: round(v, 1) for k, v in boj.items()}, " 日銀券+預け金:", round(boj["日銀券等"] + boj["日銀預け金"], 1))

表7 予想外の消費者物価による部門別の損得（兆円、＋＝得）


         NNP(2021年度末,狭義)  損得(狭義)  損得(広義)  2003〜21(狭義)  2003〜21(広義)
日本銀行                -3.1     0.2     0.2          0.8          0.8
預金取扱機関            -194.2     9.8     9.6         -7.4         -7.3
保険・年金基金            340.8   -15.2    10.1         21.8         -8.1
家計等                803.8   -37.3   -62.7         41.3         72.8
民間非金融法人           -112.0     5.6     6.0         -7.4         -9.2
一般政府             -1151.0    52.3    52.3        -60.0        -60.0
海外                 237.1   -11.2   -11.3          8.6          8.6
その他                 78.6    -4.2    -4.2          2.4          2.4
合計                   0.0     0.0     0.0         -0.0          0.0
家計等の2022〜25の損得を項目別に(兆円): {'現金・預金': -52.42, '貸出': 17.19, '債務証券': -1.56, 'その他(預け金・企業間信用・未収未払)': -0.48}  合計: -37.26

家計の2025年度末：現金・預金(資産−負債) 1171  貸出(借入) -405
日銀の2025年度末: {'債務証券': 487.6, '貸出': 78.1, '日銀券等': 120.9, '日銀預け金': 459.7}  日銀券+預け金: 580.7


## 5. 統合政府（7節、表8、図5）

In [9]:
# 8. 統合政府（一般政府＋日銀）：NNP、外部負債の構成、金利感応度
con = narrow["一般政府"] + narrow["日本銀行"]
print("統合政府NNP(兆円):", con.loc[[2021, 2025]].round(1).to_dict(), " 対名目GDP(%):", (con / Y).loc[[2021, 2025]].mul(100).round(1).to_dict(),
      " 一般政府のみ2025:", round(float(narrow['一般政府'][2025]), 1))
print("FY2022〜25の損得(兆円) 一般政府/日銀/統合:", round(float(G1.loc[2022:2025, '一般政府'].sum()), 1),
      round(float(G1.loc[2022:2025, '日本銀行'].sum()), 1), round(float((G1['一般政府'] + G1['日本銀行']).loc[2022:2025].sum()), 1))
rows8 = {}
for y in range(2002, 2026):
    jb = S("420", "L", "311")[y] - S("110", "A", "311")[y] - S("420", "A", "311")[y]      # 民間・海外が持つ国債・財投債
    tb = S("420", "L", "310")[y] - S("110", "A", "310")[y] - S("420", "A", "310")[y]      # 同、国庫短期証券
    cur, res, loan = S("110", "L", "110")[y], S("110", "L", "120")[y], S("420", "L", "200")[y]
    tot_ = jb + tb + cur + res + loan
    rows8[y] = {"国債・財投債": jb, "国庫短期証券": tb, "日銀券等": cur, "日銀預け金": res, "借入": loan, "計": tot_,
                "預け金＋短期証券": res + tb, "割合(%)": (res + tb) / tot_ * 100}
t8 = pd.DataFrame(rows8).T
t8["政府の他の債務証券"] = [float(S("420", "L", "300")[y] - S("420", "L", "311")[y] - S("420", "L", "310")[y]) for y in t8.index]
t8["日銀の借入"] = [float(S("110", "L", "200")[y]) for y in t8.index]
t8["割合・拡大分母(%)"] = t8["預け金＋短期証券"] / (t8["計"] + t8["政府の他の債務証券"] + t8["日銀の借入"]) * 100
print("\n表8 統合政府の主要負債項目の構成（兆円、時価）"); print(t8.loc[[2012, 2019, 2021, 2025]].round(1).to_string())
print("\n短期金利+1ptでの追加利払い(兆円/年): FY2025構造", round(float(t8.loc[2025, '預け金＋短期証券'] * 0.01), 1), " FY2021構造", round(float(t8.loc[2021, '預け金＋短期証券'] * 0.01), 1))
cpn = C[2025]; jgb_boj = S("110", "A", "311")[2025]; res25 = S("110", "L", "120")[2025]
print(f"日銀の損益分岐の目安: 国債{jgb_boj:.0f}兆円×利率{cpn*100:.2f}% = 預け金{res25:.0f}兆円×付利 → 付利 ≈ {cpn*jgb_boj/res25*100:.2f}%")
gain52 = float(G1.loc[2022:2025, '一般政府'].sum())
print("利得の総額÷1ptの追加利払い(年):", round(gain52 / float(t8.loc[2025, '預け金＋短期証券'] * 0.01), 1), "年分")

統合政府NNP(兆円): {2021: -1154.1, 2025: -1052.8}  対名目GDP(%): {2021: -200.3, 2025: -156.5}  一般政府のみ2025: -1022.1
FY2022〜25の損得(兆円) 一般政府/日銀/統合: 52.3 0.2 52.5

表8 統合政府の主要負債項目の構成（兆円、時価）
      国債・財投債  国庫短期証券   日銀券等  日銀預け金     借入       計  預け金＋短期証券  割合(%)  政府の他の債務証券  日銀の借入  割合・拡大分母(%)
2012   525.8   115.2   87.9   58.1  164.9   951.9     173.3   18.2       72.6   14.5        16.7
2019   406.2    85.8  114.6  395.3  153.4  1155.3     481.1   41.6       75.0   24.1        38.4
2021   397.3   140.4  124.9  563.2  153.5  1379.1     703.5   51.0       76.6    0.9        48.3
2025   372.2   123.2  120.9  459.7  146.2  1222.3     582.9   47.7       64.5   18.0        44.7

短期金利+1ptでの追加利払い(兆円/年): FY2025構造 5.8  FY2021構造 7.0
日銀の損益分岐の目安: 国債485兆円×利率0.98% = 預け金460兆円×付利 → 付利 ≈ 1.03%
利得の総額÷1ptの追加利払い(年): 9.0 年分


## 6. 図1〜5（白黒印刷用）

In [10]:
# 9. 図（白黒印刷用：濃淡・ハッチング・線種・マーカーで区別）
plt.rcParams.update({"font.family": ["Yu Gothic", "Meiryo", "BIZ UDGothic", "Noto Sans CJK JP"], "axes.unicode_minus": False,
                     "font.size": 13, "hatch.linewidth": 0.9, "axes.linewidth": 1.0})
K, DG, MG, LG = "#000000", "#4d4d4d", "#8c8c8c", "#d0d0d0"
def clean(ax):
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#bbbbbb", lw=0.6, ls=":"); ax.set_axisbelow(True); ax.tick_params(labelsize=12)
def save(fig, name): fig.tight_layout(); fig.savefig(FIG / name, dpi=250); plt.close(fig)
dd = d.loc[2003:2025]
# 図1
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(dd.index, dd["pi"] * 100, color=K, lw=2.6, marker="o", ms=6, mfc=K, label="実績")
ax.plot(dd.index, dd["pie"] * 100, color=DG, lw=2.0, ls="--", marker="s", ms=6, mfc="white", mew=1.6, label="政府経済見通し")
ax.plot(pie_esp.index, pie_esp * 100, color=K, lw=1.8, ls=":", marker="^", ms=8, mfc="white", mew=1.6, label="ESPフォーキャスト（1月調査）")
ax.axhline(0, color="#333", lw=0.9); clean(ax); ax.set_ylabel("GDPデフレーターの前年度比（％）"); ax.set_xlabel("年度")
ax.legend(frameon=True, edgecolor="#888", fontsize=12, loc="upper left"); save(fig, "図1_デフレーターの実績と予想.png")
# 図2
style = {"利払い": dict(fc=LG, hatch=""), "事前の名目成長": dict(fc="white", hatch="///"), "予想外の物価": dict(fc=K, hatch=""),
         "予想外の実質成長": dict(fc=MG, hatch="..."), "赤字等（残差）": dict(fc=DG, hatch="\\\\")}
fig, ax = plt.subplots(figsize=(8.6, 5.2)); pos = np.zeros(len(comp)); neg = np.zeros(len(comp))
for c_ in style:
    v = comp[c_].values; bottom = np.where(v >= 0, pos, neg)
    ax.bar(comp.index, v, bottom=bottom, label=c_, width=0.78, edgecolor="black", linewidth=0.7, **style[c_])
    pos += np.where(v >= 0, v, 0); neg += np.where(v < 0, v, 0)
ax.plot(comp.index, comp["債務比率の変化"], color="black", marker="D", ms=6, mfc="white", mew=1.6, lw=1.6, label="債務比率の変化")
ax.axhline(0, color="#000", lw=0.9); clean(ax); ax.set_ylabel("普通国債残高／名目GDPの変化（ポイント）"); ax.set_xlabel("年度")
ax.legend(frameon=True, edgecolor="#888", ncol=3, fontsize=10.5, loc="upper left"); save(fig, "図2_債務比率の要因分解.png")
# 図3
sh2 = sh.loc[2002:2025] * 100
order = ["日本銀行", "預金取扱機関", "保険・年金基金", "家計等", "海外", "政府・公的金融", "その他"]
fcs = [K, "white", MG, "white", LG, DG, "white"]; hs = ["", "///", "", "...", "", "", "xxx"]
fig, ax = plt.subplots(figsize=(8.2, 5.0))
polys = ax.stackplot(sh2.index, [sh2[c_] for c_ in order], colors=fcs, edgecolor="black", linewidth=0.7)
for p_, h in zip(polys, hs): p_.set_hatch(h)
cum = np.zeros(len(sh2)); lab_x = {"日本銀行": 2022, "預金取扱機関": 2008, "保険・年金基金": 2009, "海外": 2018, "政府・公的金融": 2004}
for c_ in order:
    mid = cum + sh2[c_].values / 2; cum += sh2[c_].values
    if c_ in lab_x:
        dark = c_ in ("日本銀行", "政府・公的金融")
        ax.text(lab_x[c_], mid[list(sh2.index).index(lab_x[c_])], c_, ha="center", va="center", fontsize=11.5, color="white" if dark else "black",
                bbox=dict(boxstyle="round,pad=0.2", fc="none" if dark else "white", ec="none", alpha=0.9))
ax.text(2008, 76.3, "家計等", ha="center", va="center", fontsize=10.5, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
ax.text(2011, 98.6, "その他", ha="center", va="center", fontsize=10.5, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none"))
clean(ax); ax.set_ylim(0, 100); ax.set_ylabel("国債・財投債の保有シェア（％）"); ax.set_xlabel("年度末"); ax.grid(False); save(fig, "図3_国債の保有シェア.png")
# 図4
keys = ["一般政府", "預金取扱機関", "民間非金融法人", "日本銀行", "保険・年金基金", "海外", "家計等"]
labs = ["一般政府", "預金取扱\n機関", "民間非金融\n法人", "日本銀行", "保険・年金\n基金", "海外", "家計等"]
g1, g2 = G1.loc[2022:2025].sum(), G2.loc[2022:2025].sum(); x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(x - 0.2, [g1[k] for k in keys], 0.4, color=DG, edgecolor="black", linewidth=0.8, label="狭義（保険・年金を除く）")
ax.bar(x + 0.2, [g2[k] for k in keys], 0.4, color="white", edgecolor="black", hatch="///", linewidth=0.8, label="広義（保険・年金を含む）")
for xi, k in zip(x, keys):
    for off, GG in ((-0.2, g1), (0.2, g2)):
        v = GG[k]
        if abs(v) >= 4: ax.text(xi + off, v + (1.8 if v > 0 else -1.8), f"{v:+.0f}", ha="center", va="bottom" if v > 0 else "top", fontsize=10)
ax.axhline(0, color="#000", lw=0.9); ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=10.5); clean(ax); ax.set_ylim(-72, 62)
ax.set_ylabel("予想外の消費者物価による損得（兆円）"); ax.legend(frameon=True, edgecolor="#888", fontsize=11, loc="lower left"); save(fig, "図4_部門別の損得.png")
# 図5
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(t8.index, t8["割合(%)"], color="black", lw=2.4, marker="o", ms=6)
for yy in (2012, 2021, 2025): ax.annotate(f"{t8.loc[yy,'割合(%)']:.0f}%", (yy, t8.loc[yy, "割合(%)"]), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=11)
clean(ax); ax.set_ylabel("選定した負債に占める短期金利連動分（％）"); ax.set_xlabel("年度末"); ax.set_ylim(0, 62); save(fig, "図5_短期金利連動の割合.png")
print("図1〜5を保存:", FIG.relative_to(ROOT))

findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


...

findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


findfont: Font family 'Noto Sans CJK JP' not found.


図1〜5を保存: output\figures


## 7. 本文の数値との突合せ

論文本文に書かれた数値と、このノートブックの出力が一致するかを判定する。許容差は、本文の表記（小数第1位または第2位）の丸め幅に合わせてある。`NG` が出た場合は、本文かノートブックのどちらかを直す。

In [11]:
# 10. 本文の数値との突合せ
CHECKS.clear()
b = d["b"] * 100
# 1節・4節
chk("1節 2021年度末の債務比率(%)", b[2021], 172.1, 0.06); chk("1節 2022年度末の債務比率(%)", b[2022], 173.6, 0.06); chk("1節 2025年度末の債務比率(%)", b[2025], 164.2, 0.06)
chk("4節 2002年度末の債務比率(%)", d.loc[2003, "bprev"] * 100, 79.4, 0.06)
chk("1節 2021年度末の普通国債残高(兆円)", B[2021], 991, 0.5); chk("1節 2025年度末の普通国債残高(兆円)", B[2025], 1104, 0.5); chk("1節 残高の増加(兆円)", B[2025] - B[2021], 113, 0.5)
for y, v in [(2023, 4.8), (2024, 3.2), (2025, 3.3)]: chk(f"1節 {y}年度のデフレーター(%)", d.loc[y, "pi"] * 100, v, 0.05)
chk("4節 2023年度の見通し(%)", d.loc[2023, "pie"] * 100, 0.6, 0.05)
r22 = t2.loc["2022〜2025年度"]; r03 = t2.loc["2003〜2021年度"]; r25 = t2.loc["2003〜2025年度"]
for k, v in {"債務比率の変化": -7.9, "利払い": 5.3, "事前の名目成長": -18.7, "予想外の物価": -13.9, "予想外の実質成長": 6.8, "赤字等（残差）": 12.7}.items(): chk(f"表2 2022〜25 {k}", r22[k], v, 0.06)
for k, v in {"債務比率の変化": 92.7, "利払い": 27.8, "事前の名目成長": -50.4, "予想外の物価": 11.5, "予想外の実質成長": 25.0, "赤字等（残差）": 78.8}.items(): chk(f"表2 2003〜21 {k}", r03[k], v, 0.06)
for k, v in {"債務比率の変化": 84.8, "利払い": 33.1, "事前の名目成長": -69.2, "予想外の物価": -2.4, "予想外の実質成長": 31.8, "赤字等（残差）": 91.5}.items(): chk(f"表2 2003〜25 {k}", r25[k], v, 0.06)
chk("4節 予想外を除いた2022〜25の変化(pt)", noshock, -0.7, 0.06); chk("4節 2023年度の予想外の物価の寄与(pt)", comp.loc[2023, "予想外の物価"], -7.0, 0.06)
for (r_, c_), v in {(0, 0): 171.3, (0, 1): 172.2, (0, 2): 173.2, (1, 0): 178.1, (1, 1): 180.6, (1, 2): 183.0}.items(): chk(f"表3 {r_}-{c_}", t3.iloc[r_, c_], v, 0.06)
# 5節
for (a, z), (mu, tv, n) in bias.items():
    exp = {(2003, 2021): (-0.52, -3.4), (2022, 2025): (2.09, 6.6), (2003, 2025): (-0.07, -0.2)}[(a, z)]
    chk(f"表4 {a}〜{z} 平均", mu, exp[0], 0.006); chk(f"表4 {a}〜{z} t値", tv, exp[1], 0.06)
exp5 = {"政府経済見通し": (10.95, -13.91), "過去3年度の平均の実績": (-5.24, -10.09), "前年度の実績": (-1.80, -5.58)}
for k, (a, b_) in exp5.items(): chk(f"表5 {k} 2004〜21", t5.loc[k, "2004〜2021"], a, 0.006); chk(f"表5 {k} 2022〜25", t5.loc[k, "2022〜2025"], b_, 0.006)
chk("表5 ESP 2019〜21", t5.loc["民間予想（ESP）", "2004〜2021"], -1.09, 0.006); chk("表5 ESP 2022〜25", t5.loc["民間予想（ESP）", "2022〜2025"], -11.25, 0.006)
chk("表5 公表当時 2015〜21", t5.loc["政府見通し・公表当時の実績", "2004〜2021"], 7.55, 0.006); chk("表5 公表当時 2022〜25", t5.loc["政府見通し・公表当時の実績", "2022〜2025"], -11.30, 0.006)
chk("5.3節 最新 2015〜21", c_l.loc[2015:2021].sum(), 3.8, 0.06)
chk("5.2節 2004〜2025の合計 政府", t5.loc["政府経済見通し", "2004〜2025"], -2.96, 0.006)
chk("5.2節 2004〜2025の合計 前年実績", t5.loc["前年度の実績", "2004〜2025"], -7.38, 0.006); chk("5.2節 2004〜2025の合計 3年平均", t5.loc["過去3年度の平均の実績", "2004〜2025"], -15.33, 0.006)
for k, v in zip(["政府", "前年実績", "3年平均"], [0.77, 0.87, 1.05]): chk(f"5.2節 RMSE {k}", rmse[k], v, 0.006)
chk("5.4節 税収の増加率(%)", (T[2025] / T[2020] - 1) * 100, 38, 0.6); chk("5.4節 名目GDPの増加率(%)", (Y[2025] / Y[2020] - 1) * 100, 21, 0.6)
chk("5.4節 対数の比", np.log(T[2025] / T[2020]) / np.log(Y[2025] / Y[2020]), 1.7, 0.06)
# 査読後に追加した数値
chk("4.1節 交差項を物価側に配賦した場合の2022〜25の物価寄与", c_rev.loc[2022:2025].sum(), -13.77, 0.006)
chk("4.3節 弾性値0の差(7.12)", d.loc[2025, "b"] * 100 - t3.iloc[0, 0], -7.12, 0.006); chk("4.3節 会計寄与の和(7.16)", acc, -7.16, 0.006)
chk("4.3節 物価のみの差(13.94)", d.loc[2025, "b"] * 100 - t3.iloc[1, 0], -13.94, 0.006)
chk("5.3節 重みも初出にした場合 2022〜25", c_v2.loc[2022:2025].sum(), -11.7, 0.06); chk("5.3節 重みも初出にした場合 2015〜21", c_v2.loc[2015:2021].sum(), 7.9, 0.06)
chk("5.1節 CPI見通しの誤差 平均", cpi_bias[0], -0.32, 0.006); chk("5.1節 CPI見通しの誤差 t値", cpi_bias[1], -2.5, 0.06)
for k, v in {"現金・預金": -52.4, "貸出": 17.2, "債務証券": -1.6, "その他(預け金・企業間信用・未収未払)": -0.5}.items(): chk(f"6.4節 家計の損得の内訳 {k}", hh_gain[k], v, 0.06)
chk("6.4節 借入の相殺割合(約3分の1)", hh_gain["貸出"] / -hh_gain["現金・預金"], 1 / 3, 0.02)
for y, v in {2012: 16.7, 2021: 48.3, 2025: 44.7}.items(): chk(f"7.2節 拡大分母での割合 {y}", t8.loc[y, "割合・拡大分母(%)"], v, 0.06)
chk("7.2節 政府のその他の債務証券 2025", t8.loc[2025, "政府の他の債務証券"], 64.5, 0.06); chk("7.2節 日銀の借入 2025", t8.loc[2025, "日銀の借入"], 18.0, 0.06)
# 6節
sp = sh * 100
for k, v in {"預金取扱機関": 32, "保険・年金基金": 21, "政府・公的金融": 20, "日本銀行": 15}.items(): chk(f"6.1節 2002年度末 {k}", sp.loc[2002, k], v, 0.6)
for k, v in {"日本銀行": 48, "預金取扱機関": 13, "家計等": 2.3, "海外": 7.9}.items(): chk(f"6.1節 2025年度末 {k}", sp.loc[2025, k], v, 0.06 if v < 10 else 0.6)
for k, v in {"日本銀行": 23.7, "預金取扱機関": 5.6, "保険・年金基金": 10.3, "家計等": 0.7, "海外": 3.1, "政府・公的金融": 2.3, "その他": 1.0, "合計": 46.8}.items(): chk(f"表6 2022〜25 {k}", t6.loc[k, "2022〜2025年度"], v, 0.06)
for k, v in {"日本銀行": -17.0, "預金取扱機関": -11.3, "保険・年金基金": -12.1, "家計等": -1.0, "海外": -2.7, "政府・公的金融": -3.0, "その他": -1.2, "合計": -48.2}.items(): chk(f"表6 2003〜21 {k}", t6.loc[k, "2003〜2021年度"], v, 0.06)
chk("6.2節 民間と海外の直接保有者の損失", priv, 20.8, 0.06); chk("6.2節 デフレーター基準の国の利得(兆円)", defl_gain, 84, 0.6); chk("6.2節 CPI基準の保有者の損失", t6.loc["合計", "2022〜2025年度"], 47, 0.6)
chk("6.2節 倍率", defl_gain / t6.loc["合計", "2022〜2025年度"], 1.8, 0.06)
exp7 = {"一般政府": (-1151.0, 52.3, 52.3, -60.0, -60.0), "預金取扱機関": (-194.2, 9.8, 9.6, -7.4, -7.3), "民間非金融法人": (-112.0, 5.6, 6.0, -7.4, -9.2), "日本銀行": (-3.1, 0.2, 0.2, 0.8, 0.8),
        "保険・年金基金": (340.8, -15.2, 10.1, 21.8, -8.1), "海外": (237.1, -11.2, -11.3, 8.6, 8.6), "家計等": (803.8, -37.3, -62.7, 41.3, 72.8), "その他": (78.6, -4.2, -4.2, 2.4, 2.4)}
for k, vs in exp7.items():
    for c_, v in zip(t7.columns, vs): chk(f"表7 {k} {c_}", t7.loc[k, c_], v, 0.06)
chk("6.4節 家計の現金・預金(兆円)", hh["100"], 1171, 0.6); chk("6.4節 家計の借入(兆円)", -hh["200"], 405, 0.6)
chk("6.4節 日銀の債務証券(兆円)", boj["債務証券"], 488, 0.6); chk("6.4節 日銀の貸出(兆円)", boj["貸出"], 78, 0.6); chk("6.4節 日銀券・当座預金(兆円)", boj["日銀券等"] + boj["日銀預け金"], 581, 0.6)
# 7節
chk("7.1節 統合NNP 2025(兆円)", con[2025], -1052.8, 0.06); chk("7.1節 統合NNP/GDP 2025(%)", (con / Y)[2025] * 100, -156.5, 0.06); chk("7.1節 一般政府NNP 2025", narrow["一般政府"][2025], -1022.1, 0.06)
chk("7.1節 利得 一般政府", G1.loc[2022:2025, "一般政府"].sum(), 52.3, 0.06); chk("7.1節 利得 日銀", G1.loc[2022:2025, "日本銀行"].sum(), 0.2, 0.06); chk("7.1節 利得 統合", (G1["一般政府"] + G1["日本銀行"]).loc[2022:2025].sum(), 52.5, 0.06)
exp8 = {2012: (525.8, 115.2, 87.9, 58.1, 164.9, 951.9, 18.2), 2019: (406.2, 85.8, 114.6, 395.3, 153.4, 1155.3, 41.6), 2021: (397.3, 140.4, 124.9, 563.2, 153.5, 1379.1, 51.0), 2025: (372.2, 123.2, 120.9, 459.7, 146.2, 1222.3, 47.7)}
for y, vs in exp8.items():
    for c_, v in zip(["国債・財投債", "国庫短期証券", "日銀券等", "日銀預け金", "借入", "計", "割合(%)"], vs): chk(f"表8 {y} {c_}", t8.loc[y, c_], v, 0.06)
chk("7.2節 割合 2012", t8.loc[2012, "割合(%)"], 18, 0.6); chk("7.2節 割合 2021", t8.loc[2021, "割合(%)"], 51, 0.6); chk("7.2節 割合 2025", t8.loc[2025, "割合(%)"], 48, 0.6)
chk("7.3節 +1ptの追加利払い 2025構造", t8.loc[2025, "預け金＋短期証券"] * 0.01, 5.8, 0.06); chk("7.3節 +1ptの追加利払い 2021構造", t8.loc[2021, "預け金＋短期証券"] * 0.01, 7.0, 0.06)
chk("7.3節 付利の目安(%)", cpn * jgb_boj / res25 * 100, 1.0, 0.06)
chk("7.3節 利得÷追加利払い(年)", gain52 / (t8.loc[2025, "預け金＋短期証券"] * 0.01), 9, 0.6)
# 3.2節（財務省の利払費の決算値との比較。決算値は財務省「利払費と金利の推移」のグラフの数値）
for y, (ap, act) in {2005: (7.4, 7.0), 2010: (7.9, 7.9), 2015: (8.5, 8.3), 2020: (7.5, 7.4)}.items():
    i_ = (C[y] + C[y - 1]) / 2; chk(f"3.2節 利払いの近似 {y}", i_ * B[y - 1], ap, 0.06)
res = pd.DataFrame(CHECKS, columns=["内容", "ノートブック", "本文", "許容差"]); res["判定"] = np.where((res["ノートブック"] - res["本文"]).abs() <= res["許容差"] + 1e-9, "OK", "NG")
print(f"突合せ {len(res)} 件：OK {int((res['判定']=='OK').sum())} 件、NG {int((res['判定']=='NG').sum())} 件")
pd.set_option("display.width", 200, "display.max_rows", 500, "display.max_colwidth", 60)
print(res[res["判定"] == "NG"].round(3).to_string() if (res["判定"] == "NG").any() else "すべて一致")
res.to_csv(TAB / "numbers_check.csv", index=False, encoding="utf-8-sig")

突合せ 199 件：OK 199 件、NG 0 件
すべて一致


## 付録：再現環境

実行した環境は次のセルで表示する。

In [12]:
import sys, platform
print("Python", sys.version.split()[0], "/ pandas", pd.__version__, "/ numpy", np.__version__, "/ matplotlib", matplotlib.__version__, "/", platform.platform())

Python 3.12.10 / pandas 3.0.5 / numpy 2.5.2 / matplotlib 3.11.1 / Windows-11-10.0.26100-SP0
